In [ ]:
!pip install langchain langchain-community langchain-openai openai tiktoken python-dotenv wikipedia faiss-cpu langchain-google-genai beautifulsoup4 langgraph notion-client
!npm install localtunnel

In [ ]:
!curl ipv4.icanhazip.com

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
os.environ['OPENAI_API_KEY'] = user_secrets.get_secret("OPENAI_API_KEY")
os.environ['GOOGLE_API_KEY'] = user_secrets.get_secret("GOOGLE_API_KEY")
os.environ['TAVILY_API_KEY'] = user_secrets.get_secret("TAVILY_API_KEY")
os.environ['NOTION_TOKEN'] = user_secrets.get_secret("NOTION_TOKEN")
os.environ['NOTION_DATABASE_ID'] = user_secrets.get_secret("NOTION_DATABASE_ID")
os.environ['GOOGLE_PAGESPEED_KEY'] = user_secrets.get_secret("GOOGLE_PAGESPEED_KEY")
os.environ["USER_AGENT"] = "GeoAuditBot/1.0 (TFG Student)"


In [ ]:
import os
import re
import time
import json
import requests  
import operator
from datetime import datetime
from typing import Annotated, List, TypedDict

# --- IMPORTS HERRAMIENTAS ---
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# --- IMPORTS IA ---
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- IMPORT NOTION ---
from notion_client import Client

# --- IMPORTS GRAFO ---
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv

# Cargar entorno
load_dotenv()

# --- 1. ESTADO ---
class GeoAuditState(TypedDict):
    target_url: str
    generated_prompts: List[str]
    competitor_urls: Annotated[List[str], operator.add]
    documents: List[Document] 
    rag_results: List[dict]
    # NUEVO: Aquí guardaremos las métricas técnicas
    seo_metrics: dict 
    logs: List[str]

# --- 2. NODO A: ESTRATEGA (CORREGIDO) ---
def strategist_node(state: GeoAuditState):
    print(f"--- [Nodo A] Investigando marca: {state['target_url']} ---")
    
    # Investigación previa
    search_tool = TavilySearchResults(max_results=5)
    try:
        query = f"Qué servicios ofrece {state['target_url']} y cuál es su propuesta de valor?"
        # Convertimos a string para que sea texto plano
        context_str = str(search_tool.invoke({"query": query}))
    except:
        context_str = "Información no disponible."

    # Generación de prompts
    llm = ChatOpenAI(model="gpt-4o", temperature=0.7)
    
    system_prompt = (
        "Eres un Consultor GEO. Genera 5 PREGUNTAS ESTRATÉGICAS que usuarios harían "
        "en buscadores IA y donde ESTA marca debería aparecer.\n"
        "REGLAS: No uses el nombre de la marca. Ataca el problema del usuario.\n"
        "Devuelve solo la lista, una por línea."
    )
    
    # CORRECCIÓN AQUÍ:
    # 1. No usamos f-string en el template. Usamos {info_marca} como placeholder.
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt), 
        ("human", "Info marca: {info_marca}") 
    ])
    
    chain = prompt_template | llm
    
    # 2. Pasamos la variable 'info_marca' dentro del invoke
    response = chain.invoke({"info_marca": context_str})
    
    prompts = [line.strip() for line in response.content.split('\n') if line and "?" in line][:5]
    
    print(f"--- Prompts: {prompts} ---")
    return {"generated_prompts": prompts}

# --- 3. NODO B: DESCUBRIDOR ---
def discovery_node(state: GeoAuditState):
    print("--- [Nodo B] Buscando Competencia (Tavily) ---")
    prompts = state.get("generated_prompts", [])
    competitors = set()
    search_tool = TavilySearchResults(max_results=5, search_depth="advanced")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) 

    for prompt in prompts:
        try:
            results = search_tool.invoke({"query": prompt})
            selection = llm.invoke(f"Saca las 3 mejores URLs de aquí (solo URLs): {results}")
            urls = [u.strip() for u in selection.content.split('\n') if "http" in u]
            for url in urls:
                clean = url.split("](")[-1].replace(")", "").strip()
                if clean and state["target_url"] not in clean:
                    competitors.add(clean)
        except: pass

    unique = list(competitors)[:5]
    print(f"--- Competidores: {unique} ---")
    return {"competitor_urls": unique}

# --- 4. NODO C: PROCESADOR ---
def processor_node(state: GeoAuditState):
    print("--- [Nodo C] Descargando contenido ---")
    urls = [state["target_url"]] + state["competitor_urls"]
    processed_docs = []
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

    for url in urls:
        try:
            loader = WebBaseLoader(url)
            loader.requests_kwargs = {'headers': {'User-Agent': 'Mozilla/5.0'}}
            docs = loader.load()
            chunks = splitter.split_documents(docs)
            for c in chunks:
                c.metadata["source_url"] = url
                c.metadata["is_target"] = (url == state["target_url"])
            processed_docs.extend(chunks)
        except: pass
            
    return {"documents": processed_docs}

# --- 5. NODO TÉCNICO: ANALISTA SEO (NUEVO) ---
def technical_seo_node(state: GeoAuditState):
    url = state['target_url']
    print(f"--- [Nodo Técnico] Analizando PageSpeed para: {url} ---")
    
    # Recuperamos la clave usando os.getenv (Así conecta con tu código de Kaggle)
    api_key = os.getenv("GOOGLE_PAGESPEED_KEY")
    
    if not api_key:
        print("   ⚠️ ERROR: No se encontró GOOGLE_PAGESPEED_KEY en las variables de entorno.")
        return {"seo_metrics": {}}

    endpoint = "https://www.googleapis.com/pagespeedonline/v5/runPagespeed"
    params = {
        'url': url,
        'key': api_key,
        'category': ['performance', 'accessibility', 'best-practices', 'seo']
    }
    
    metrics = {}
    try:
        response = requests.get(endpoint, params=params)
        if response.status_code == 200:
            data = response.json()
            cats = data['lighthouseResult']['categories']
            audits = data['lighthouseResult']['audits']
            
            metrics = {
                "seo_score": cats['seo']['score'] * 100,
                "performance": cats['performance']['score'] * 100,
                "accessibility": cats['accessibility']['score'] * 100,
                "best_practices": cats['best-practices']['score'] * 100,
                "LCP": audits['largest-contentful-paint']['displayValue'],
                "TBT": audits['total-blocking-time']['displayValue']
            }
            
            # Guardamos JSON local
            filename = f"seo_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            with open(filename, 'w') as f:
                json.dump(metrics, f, indent=4)
            print(f"   ✅ JSON guardado: {filename}")
        else:
            print(f"   ⚠️ Error API Google: {response.status_code}")
            
    except Exception as e:
        print(f"   ⚠️ Error en Nodo Técnico: {e}")

    return {"seo_metrics": metrics}

# --- 6. NODO D: JUEZ (MÉTRICAS GEO) ---
def rag_simulator_node(state: GeoAuditState):
    print("--- [Nodo D] Juez y Métricas GEO ---")
    docs = state.get("documents", [])
    if not docs: return {"rag_results": []}

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = FAISS.from_documents(docs, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    llm = ChatOpenAI(model="gpt-4o", temperature=0)

    rag_prompt = """
    Actúa como Perplexity AI. Responde usando SOLO el contexto.
    OBLIGATORIO: Cita fuentes así [Fuente: URL].
    Contexto: {context}
    Pregunta: {question}
    """
    
    def format_docs(docs):
        return "\n\n".join(f"[URL: {d.metadata['source_url']}] {d.page_content}" for d in docs)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | ChatPromptTemplate.from_template(rag_prompt) | llm | StrOutputParser()
    )

    results = []
    for q in state["generated_prompts"]:
        print(f"   > Evaluando: {q}")
        try:
            ans = chain.invoke(q)
            citations = re.findall(r'\[Fuente: (.*?)\]', ans)
            is_visible = any(state["target_url"] in c for c in citations)
            my_cits = sum(1 for c in citations if state["target_url"] in c)
            total = len(citations)
            som = (my_cits / total * 100) if total > 0 else 0
            
            rank = None
            for i, c in enumerate(citations):
                if state["target_url"] in c:
                    rank = i + 1
                    break
            
            results.append({
                "prompt": q, "answer": ans, "visible": is_visible, 
                "som": som, "rank": rank
            })
        except Exception as e:
            print(f"Error: {e}")

    return {"rag_results": results}

# --- 7. NODO E: REPORTERO (NOTION + SEO) ---
def reporter_node(state: GeoAuditState):
    print("--- [Nodo E] Guardando informe completo en Notion ---")
    
    token = os.getenv("NOTION_TOKEN")
    db_id = os.getenv("NOTION_DATABASE_ID")
    
    # Recuperamos las métricas técnicas
    seo = state.get("seo_metrics", {})
    
    notion = Client(auth=token)
    results = state.get("rag_results", [])
    
    for r in results:
        try:
            properties = {
                "Name": {"title": [{"text": {"content": r["prompt"]}}]},
                "Respuesta": {"rich_text": [{"text": {"content": r["answer"][:2000]}}]},
                "Visibilidad": {"checkbox": r["visible"]},
                "SoM": {"number": r["som"]},
                "Ranking": {"number": r["rank"] if r["rank"] else 0},
                "SEO Score": {"number": seo.get("seo_score", 0)},
                "Performance": {"number": seo.get("performance", 0)},
                "LCP": {"rich_text": [{"text": {"content": str(seo.get("LCP", "N/A"))}}]},
                "TBT": {"rich_text": [{"text": {"content": str(seo.get("TBT", "N/A"))}}]},
                "Fecha": {"date": {"start": datetime.now().isoformat()}}

            }
            
            notion.pages.create(
                parent={"database_id": db_id},
                properties=properties
            )
            print(f"   ✅ Guardado en Notion: {r['prompt'][:30]}...")
        except Exception as e:
            print(f"   ❌ Error guardando en Notion: {e}")
            
    return {}

# --- 8. GRAFO ---
def build_geo_graph():
    wf = StateGraph(GeoAuditState)
    wf.add_node("strategist", strategist_node)
    wf.add_node("discovery", discovery_node)
    wf.add_node("processor", processor_node)
    wf.add_node("technical_seo", technical_seo_node) # <--- Nodo Nuevo
    wf.add_node("rag_simulator", rag_simulator_node)
    wf.add_node("reporter", reporter_node)

    wf.set_entry_point("strategist")
    wf.add_edge("strategist", "discovery")
    wf.add_edge("discovery", "processor")
    
    # Ejecutamos el SEO Técnico en paralelo o secuencial
    # Aquí lo pongo secuencial después de descargar, antes del RAG
    wf.add_edge("processor", "technical_seo") 
    wf.add_edge("technical_seo", "rag_simulator")
    
    wf.add_edge("rag_simulator", "reporter")
    wf.add_edge("reporter", END)

    return wf.compile()

# --- EJECUCIÓN ---
if __name__ == "__main__":
    app = build_geo_graph()
    
    initial = {
        "target_url": "https://programamos.es",
        "generated_prompts": [], "competitor_urls": [], "documents": [], 
        "rag_results": [], "seo_metrics": {}, "logs": []
    }
    
    print(">>> 🚀 AUDITORÍA GEO TOTAL (IA + TÉCNICA) <<<")
    final = app.invoke(initial)
    print("\n✅ Proceso completado.")